In [ ]:
"""实验2：卷积去噪 —— 算法版（卷积、噪声、滤波全部从零实现）"""
import tkinter as tk
from tkinter import ttk, filedialog
import numpy as np
import cv2  # 仅用于 I/O
import matplotlib
matplotlib.use('TkAgg')
matplotlib.rcParams['font.sans-serif'] = ['SimHei','Microsoft YaHei','Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk

# ============================================================
#                       算法核心部分
# ============================================================

def pad_image(img, pH, pW, mode='reflect'):
    """
    图像填充（三种边界模式）
    -------------------------------------------------
    pH, pW : 行 / 列方向的填充半径
    mode   : 'zero'    -- 用 0 填
             'edge'    -- 用最近边界像素的值填
             'reflect' -- 关于边界对称地反射
    """
    H, W = img.shape
    out = np.zeros((H+2*pH, W+2*pW), dtype=np.float32)  # 先建一个全 0 大图
    out[pH:pH+H, pW:pW+W] = img                          # 中心填入原图
    if mode == 'zero':
        return out                                       # 周围保持为 0
    if mode == 'edge':
        out[:pH, pW:pW+W] = img[0:1, :]                  # 上边
        out[pH+H:, pW:pW+W] = img[H-1:H, :]              # 下边
        out[:, :pW] = out[:, pW:pW+1]                    # 左边
        out[:, pW+W:] = out[:, pW+W-1:pW+W]              # 右边
        return out
    # reflect 模式
    out[:pH, pW:pW+W] = img[pH:0:-1, :]                  # 上边反射
    out[pH+H:, pW:pW+W] = img[H-2:H-2-pH:-1, :]          # 下边反射
    out[:, :pW] = out[:, 2*pW:pW:-1]                     # 左边反射
    out[:, pW+W:] = out[:, pW+W-2:W-2:-1]                # 右边反射
    return out


def convolve2d(img, kernel, pad_mode='reflect'):
    """
    二维卷积（实际为相关运算，对称核两者结果相同）
    -------------------------------------------------
    步骤：
      1) 把图像四周填充 pH=kH//2 行、pW=kW//2 列；
      2) 对每个核位置 (m,n)，把"平移后的整张图"乘以 kernel[m,n] 累加；
         这等价于：output[i,j] = Σ_{m,n} kernel[m,n]·padded[i+m,j+n]
         只是把内层"对每个像素分别计算"换成了"对每个核权重一次性算完整图"，
         效率高很多但语义完全一样。
    """
    H, W = img.shape
    kH, kW = kernel.shape
    pH, pW = kH // 2, kW // 2                       # 1) 填充半径
    padded = pad_image(img.astype(np.float32), pH, pW, pad_mode)
    out = np.zeros((H, W), dtype=np.float32)        # 输出累加器
    for m in range(kH):                             # 2) 遍历核的每个位置
        for n in range(kW):
            w = kernel[m, n]                        # 当前核权重
            # padded[m:m+H, n:n+W] 就是"把图向左上平移 (m,n) 的结果"
            out += w * padded[m:m+H, n:n+W]         # 加权累加
    return out


def make_gaussian_kernel(k, sigma):
    """
    二维高斯核   G(x,y) = exp( -(x²+y²)/(2σ²) ) / Z
    -------------------------------------------------
    k     : 核大小（奇数）
    sigma : 标准差，越大越平滑
    Z     : 归一化系数（让核权重之和 = 1）
    """
    r = k // 2                                       # 半径
    kernel = np.zeros((k, k), dtype=np.float32)      # 初始化全 0
    for i in range(k):                               # 双层循环填核
        for j in range(k):
            x = i - r                                # 相对中心的偏移
            y = j - r
            kernel[i, j] = np.exp(-(x*x + y*y) / (2 * sigma * sigma))
    kernel = kernel / kernel.sum()                   # 归一化（权重之和=1）
    return kernel


def add_gaussian_noise(img, sigma, mean=0.0):
    """
    添加高斯白噪声  I' = I + N(mean, σ²)
    """
    noise = np.random.randn(*img.shape) * sigma + mean   # 服从 N(mean,σ²) 的随机数
    out = img.astype(np.float32) + noise                 # 直接相加
    return np.clip(out, 0, 255).astype(np.uint8)         # 截断回 [0,255]


def add_salt_pepper(img, p, salt_ratio=0.5):
    """
    添加椒盐噪声
    -------------------------------------------------
    p           : 噪声总占比，比如 0.05 表示 5% 像素被污染
    salt_ratio  : 这些噪声中"盐"(白点) 占多少，剩下是"椒"(黑点)
    """
    out = img.copy()
    rnd = np.random.rand(*img.shape)                    # 每个像素一个 [0,1] 随机数
    salt_thr = p * salt_ratio                           # 盐阈值
    pepper_thr = p                                      # 椒阈值
    out[rnd < salt_thr] = 255                           # 小于盐阈值 → 白点
    out[(rnd >= salt_thr) & (rnd < pepper_thr)] = 0     # 落在椒区间 → 黑点
    return out


def median_filter(img, k, pad_mode='reflect'):
    """
    中值滤波（非线性，对椒盐噪声极有效）
    -------------------------------------------------
    步骤：对每个像素，取 k×k 邻域里的所有像素，
         排序后取"中间那个"作为输出值。
    """
    H, W = img.shape
    r = k // 2
    padded = pad_image(img.astype(np.float32), r, r, pad_mode)
    out = np.zeros_like(img)
    half = (k * k) // 2                              # 中位数在排序后的位置
    for i in range(H):                               # 双层循环逐像素
        for j in range(W):
            patch = padded[i:i+k, j:j+k].ravel()     # 取邻域并展平成 1D
            patch_sorted = np.sort(patch)            # 排序
            out[i, j] = patch_sorted[half]           # 取中位数
    return out.astype(np.uint8)


def mean_kernel(k):
    """均值核：所有元素都是 1/(k·k)，权重之和=1"""
    return np.ones((k, k), dtype=np.float32) / (k * k)


def psnr(a, b):
    """峰值信噪比，越大表示两图越接近"""
    mse = np.mean((a.astype(np.float32) - b.astype(np.float32)) ** 2)
    return 99.0 if mse < 1e-6 else 10 * np.log10(255 ** 2 / mse)

# ============================================================
#                       GUI（与原版一致）
# ============================================================
BG_DARK='#1e1e2e'; BG_PANEL='#252537'; BG_LIGHT='#f7f7fa'
FG_TEXT='#e4e4ef'; FG_MUTED='#9090a8'; ACCENT='#7c9cff'; ACCENT_2='#ff8a8a'

def make_demo_image():
    img = np.zeros((200,200), np.uint8)
    cv2.rectangle(img,(30,30),(95,95),200,-1)
    cv2.circle(img,(140,60),30,150,-1)
    cv2.rectangle(img,(50,125),(160,170),100,-1)
    return img

class LabeledSlider(tk.Frame):
    def __init__(self,master,text,frm,to,init,cb,fmt='{:.2f}',res=0.01):
        super().__init__(master,bg=BG_PANEL); self.cb=cb; self.fmt=fmt; self.res=res
        top=tk.Frame(self,bg=BG_PANEL); top.pack(fill='x',pady=(8,0))
        tk.Label(top,text=text,bg=BG_PANEL,fg=FG_TEXT,font=('Segoe UI',10,'bold')).pack(side='left')
        self.val_lbl=tk.Label(top,text=fmt.format(init),bg=BG_PANEL,fg=ACCENT,
                              font=('Consolas',10,'bold')); self.val_lbl.pack(side='right')
        self.var=tk.DoubleVar(value=init)
        ttk.Scale(self,from_=frm,to=to,variable=self.var,orient='horizontal',
                  command=self._on).pack(fill='x',pady=(2,6))
    def _on(self,_):
        v=self.var.get()
        if self.res>=1: v=round(v)
        self.val_lbl.config(text=self.fmt.format(v)); self.cb()
    def get(self):
        v=self.var.get(); return round(v) if self.res>=1 else v

class App:
    def __init__(self,root):
        self.root=root
        root.title('实验2 · 卷积去噪 — 算法版')
        root.geometry('1340x830'); root.configure(bg=BG_DARK)
        self.img=make_demo_image()
        self._style(); self._ui(); self._update()

    def _style(self):
        st=ttk.Style(); st.theme_use('clam')
        st.configure('TScale',background=BG_PANEL,troughcolor='#3a3a50',
                     bordercolor=BG_PANEL,lightcolor=ACCENT,darkcolor=ACCENT)
        st.configure('TCombobox',fieldbackground='#3a3a50',background='#3a3a50',
                     foreground=FG_TEXT,arrowcolor=FG_TEXT)

    def _ui(self):
        side=tk.Frame(self.root,bg=BG_PANEL,width=300); side.pack(side='left',fill='y'); side.pack_propagate(False)
        tk.Label(side,text='⚙  参数控制',bg=BG_PANEL,fg=FG_TEXT,
                 font=('Segoe UI',14,'bold')).pack(anchor='w',padx=18,pady=(20,6))
        tk.Frame(side,bg=ACCENT,height=2).pack(fill='x',padx=18)

        b=tk.Frame(side,bg=BG_PANEL); b.pack(fill='x',padx=18,pady=(14,4))
        tk.Label(b,text='噪声类型',bg=BG_PANEL,fg=FG_TEXT,font=('Segoe UI',10,'bold')).pack(anchor='w')
        self.noise=tk.StringVar(value='高斯噪声')
        ttk.Combobox(b,textvariable=self.noise,state='readonly',
                     values=['高斯噪声','椒盐噪声']).pack(fill='x',pady=(4,4))
        self.noise.trace_add('write',lambda *a: self._update())

        wrap=tk.Frame(side,bg=BG_PANEL); wrap.pack(fill='x',padx=18)
        self.nlv=LabeledSlider(wrap,'噪声强度',0,50,15,self._update,'{:.1f}',0.1); self.nlv.pack(fill='x')
        self.nmean=LabeledSlider(wrap,'高斯均值',-30,30,0,self._update,'{:.1f}',0.5); self.nmean.pack(fill='x')
        self.salt=LabeledSlider(wrap,'盐胡椒比',0,1,0.5,self._update,'{:.2f}',0.01); self.salt.pack(fill='x')

        b2=tk.Frame(side,bg=BG_PANEL); b2.pack(fill='x',padx=18,pady=(12,4))
        tk.Label(b2,text='滤波方法',bg=BG_PANEL,fg=FG_TEXT,font=('Segoe UI',10,'bold')).pack(anchor='w')
        self.flt=tk.StringVar(value='高斯')
        ttk.Combobox(b2,textvariable=self.flt,state='readonly',
                     values=['均值','高斯','中值']).pack(fill='x',pady=(4,4))
        self.flt.trace_add('write',lambda *a: self._update())

        b3=tk.Frame(side,bg=BG_PANEL); b3.pack(fill='x',padx=18,pady=(8,4))
        tk.Label(b3,text='填充模式',bg=BG_PANEL,fg=FG_TEXT,font=('Segoe UI',10,'bold')).pack(anchor='w')
        self.pad=tk.StringVar(value='reflect')
        ttk.Combobox(b3,textvariable=self.pad,state='readonly',
                     values=['zero','edge','reflect']).pack(fill='x',pady=(4,4))
        self.pad.trace_add('write',lambda *a: self._update())

        wrap2=tk.Frame(side,bg=BG_PANEL); wrap2.pack(fill='x',padx=18)
        self.k=LabeledSlider(wrap2,'核大小 k',3,11,3,self._update,'{:.0f}',1); self.k.pack(fill='x')
        self.sig=LabeledSlider(wrap2,'σ (高斯)',0.3,5.0,1.0,self._update,'{:.2f}',0.01); self.sig.pack(fill='x')

        tk.Frame(side,bg=BG_PANEL,height=10).pack()
        self._btn(side,'📁  打开图像',self._open)
        self._btn(side,'💾  保存结果',self._save)
        self._btn(side,'🔄  重置参数',self._reset, ACCENT_2)

        info=tk.Frame(side,bg='#2c2c40'); info.pack(side='bottom',fill='x',padx=12,pady=12)
        self.info=tk.Label(info,text='',bg='#2c2c40',fg=FG_MUTED,font=('Consolas',9),justify='left',anchor='w')
        self.info.pack(fill='x',padx=10,pady=10)

        main=tk.Frame(self.root,bg=BG_LIGHT); main.pack(side='right',fill='both',expand=True)
        self.fig=plt.Figure(figsize=(12,7.5),facecolor=BG_LIGHT)
        self.canvas=FigureCanvasTkAgg(self.fig,master=main)
        self.canvas.get_tk_widget().pack(fill='both',expand=True,padx=8,pady=8)
        tb=NavigationToolbar2Tk(self.canvas,main); tb.update(); tb.configure(bg=BG_LIGHT)

    def _btn(self,p,t,c,col=ACCENT):
        tk.Button(p,text=t,command=c,bg=col,fg='white',activebackground='#5a7ae0',
                  relief='flat',font=('Segoe UI',10,'bold'),cursor='hand2',pady=8
                  ).pack(fill='x',padx=18,pady=4)

    def _open(self):
        p=filedialog.askopenfilename(filetypes=[('Image','*.png *.jpg *.jpeg *.bmp')])
        if p:
            im=cv2.imread(p,cv2.IMREAD_GRAYSCALE)
            if im is not None: self.img=cv2.resize(im,(200,200)); self._update()

    def _save(self):
        p=filedialog.asksaveasfilename(defaultextension='.png')
        if p: cv2.imwrite(p,self.result)

    def _reset(self):
        for w,t in [(self.nlv,15),(self.nmean,0),(self.salt,0.5),(self.k,3),(self.sig,1.0)]:
            w.var.set(t); w.val_lbl.config(text=w.fmt.format(t))
        self.flt.set('高斯'); self.noise.set('高斯噪声'); self.pad.set('reflect'); self._update()

    def _update(self):
        np.random.seed(0)
        if self.noise.get()=='高斯噪声':
            noisy=add_gaussian_noise(self.img, self.nlv.get(), self.nmean.get())
        else:
            noisy=add_salt_pepper(self.img, self.nlv.get()/100.0, self.salt.get())

        k=int(self.k.get())|1; f=self.flt.get(); s=self.sig.get(); pad=self.pad.get()
        if f=='均值':
            kernel=mean_kernel(k); kname=f'均值核 {k}×{k}'
            self.result=np.clip(convolve2d(noisy,kernel,pad),0,255).astype(np.uint8)
        elif f=='高斯':
            kernel=make_gaussian_kernel(k,s); kname=f'高斯核 {k}×{k}  σ={s:.2f}'
            self.result=np.clip(convolve2d(noisy,kernel,pad),0,255).astype(np.uint8)
        else:
            kernel=None; kname=f'中值滤波 {k}×{k}'
            self.result=median_filter(noisy,k,pad)

        p1=psnr(self.img,noisy); p2=psnr(self.img,self.result)

        self.fig.clear()
        gs=self.fig.add_gridspec(2,3,hspace=0.4,wspace=0.32,height_ratios=[1.1,1],
                                 left=0.05,right=0.97,top=0.93,bottom=0.07)
        a=[self.fig.add_subplot(gs[i,j]) for i in range(2) for j in range(3)]
        a[0].imshow(self.img,cmap='gray'); a[0].set_title('原图',fontsize=11,fontweight='bold'); a[0].axis('off')
        a[1].imshow(noisy,cmap='gray');    a[1].set_title(f'加噪 PSNR={p1:.2f}dB',fontsize=11,fontweight='bold'); a[1].axis('off')
        a[2].imshow(self.result,cmap='gray'); a[2].set_title(f'去噪 PSNR={p2:.2f}dB',fontsize=11,fontweight='bold'); a[2].axis('off')

        if kernel is not None:
            im=a[3].imshow(kernel,cmap='viridis'); a[3].set_title(kname,fontsize=10)
            if k<=7:
                for (i,j),v in np.ndenumerate(kernel):
                    a[3].text(j,i,f'{v:.3f}',ha='center',va='center',
                              color='white' if v<kernel.max()*0.5 else 'black',fontsize=7)
            a[3].set_xticks([]); a[3].set_yticks([])
            self.fig.colorbar(im,ax=a[3],fraction=0.046,pad=0.04)
            x=np.arange(k)-k//2; c=kernel[k//2]
            a[4].plot(x,c,'o-',color='#ff7043',lw=2)
            a[4].fill_between(x,0,c,alpha=0.25,color='#ff7043')
            a[4].set_title('核中心行权重',fontsize=10); a[4].grid(alpha=0.3)
        else:
            a[3].text(0.5,0.5,'中值滤波\n邻域排序后取中位数\n(非线性,无固定核)',
                      ha='center',va='center',fontsize=11,transform=a[3].transAxes)
            a[3].set_title(kname,fontsize=10); a[3].axis('off')
            a[4].axis('off')

        residual=np.abs(noisy.astype(int)-self.result.astype(int)).astype(np.uint8)
        im2=a[5].imshow(residual,cmap='hot'); a[5].set_title('|加噪 − 去噪|',fontsize=10); a[5].axis('off')
        self.fig.colorbar(im2,ax=a[5],fraction=0.046,pad=0.04)

        self.canvas.draw()
        self.info.config(text=f'噪声: {self.noise.get()} ({self.nlv.get():.1f})\n'
                              f'滤波: {f}, k={k}\n填充: {pad}\nPSNR提升: {p2-p1:+.2f} dB')

if __name__=='__main__':
    root=tk.Tk(); App(root); root.mainloop()

C:\Users\Albert\AppData\Local\Temp\ipykernel_24648\725883566.py:309: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) SimHei.
  self.canvas.draw()
C:\anaconda3\Lib\tkinter\__init__.py:862: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) SimHei.
  func(*args)
